In [ ]:
import math
import re
from collections import Counter, defaultdict

# ============================================================
# NATURAL LANGUAGE PROCESSING SYSTEM
# Example domain:
# - Build a bigram language model
# - Define a small grammar and parse sentences
# - Add simple feature constraints
# - Analyze ambiguity
# - Perform an NLP task: sentiment classification
# ============================================================

# ------------------------------------------------------------
# Utility functions
# ------------------------------------------------------------

def tokenize(text):
    text = text.lower()
    return re.findall(r"[a-z']+|[.,!?;]", text)

def pad_sentence(tokens):
    return ["<s>"] + tokens + ["</s>"]

# ------------------------------------------------------------
# Training corpus for language modeling
# ------------------------------------------------------------

training_sentences = [
    "the cat chases the mouse",
    "the dog chases the cat",
    "the student reads the book",
    "the teacher reads the book",
    "the teacher sees the student",
    "the dog sees the cat",
    "the cat sees the mouse",
    "a student likes the book",
    "a teacher likes a student",
    "the happy student reads a book",
    "the small cat chases a mouse",
    "the old teacher sees the happy student",
    "the dog likes the small cat",
    "the teacher reads a small book",
    "the student sees the old teacher"
]

tokenized_corpus = [pad_sentence(tokenize(s)) for s in training_sentences]

# Build unigram and bigram counts
unigram_counts = Counter()
bigram_counts = Counter()
vocab = set()

for sent in tokenized_corpus:
    for tok in sent:
        unigram_counts[tok] += 1
        vocab.add(tok)
    for i in range(len(sent) - 1):
        bigram_counts[(sent[i], sent[i + 1])] += 1

vocab = sorted(vocab)
V = len(vocab)

# ------------------------------------------------------------
# Part 1: Language Modeling (23.1)
# ------------------------------------------------------------

def bigram_probability(prev_tok, tok, alpha=1.0):
    # Laplace-smoothed bigram probability
    numerator = bigram_counts[(prev_tok, tok)] + alpha
    denominator = unigram_counts[prev_tok] + alpha * V
    return numerator / denominator

def sentence_probability(sentence, alpha=1.0):
    tokens = pad_sentence(tokenize(sentence))
    prob = 1.0
    for i in range(len(tokens) - 1):
        prob *= bigram_probability(tokens[i], tokens[i + 1], alpha=alpha)
    return prob

def sentence_log_probability(sentence, alpha=1.0):
    tokens = pad_sentence(tokenize(sentence))
    total = 0.0
    for i in range(len(tokens) - 1):
        total += math.log(bigram_probability(tokens[i], tokens[i + 1], alpha=alpha))
    return total

# ------------------------------------------------------------
# Part 2 + 3 + 4: Grammar, augmented grammar, ambiguity
# Custom feature-based chart-style parser
# ------------------------------------------------------------

# Grammar with simple features:
# - noun phrase number agreement
# - subject-verb agreement
# - lexical ambiguity: "book" can be noun or verb, "old" can be adjective or noun
#
# Categories use dict-like labels:
#   NP_sg, NP_pl, VP_sg, VP_pl, S
#   N_sg, N_pl, Vt_sg, Vt_pl, Det_sg, Det_any, Adj

lexicon = {
    "the": ["Det_any"],
    "a": ["Det_sg"],

    "cat": ["N_sg"],
    "dog": ["N_sg"],
    "mouse": ["N_sg"],
    "student": ["N_sg"],
    "teacher": ["N_sg"],
    "book": ["N_sg", "Vt_base"],
    "books": ["N_pl"],
    "students": ["N_pl"],
    "teachers": ["N_pl"],
    "old": ["Adj", "N_sg"],
    "happy": ["Adj"],
    "small": ["Adj"],

    "chases": ["Vt_sg"],
    "sees": ["Vt_sg"],
    "reads": ["Vt_sg"],
    "likes": ["Vt_sg"],

    "chase": ["Vt_pl", "Vt_base"],
    "see": ["Vt_pl", "Vt_base"],
    "read": ["Vt_pl", "Vt_base"],
    "like": ["Vt_pl", "Vt_base"],
}

# Binary rules: Parent -> Left Right
binary_rules = [
    ("NP_sg", "Det_sg", "Nbar_sg"),
    ("NP_sg", "Det_any", "Nbar_sg"),
    ("NP_pl", "Det_any", "Nbar_pl"),

    ("Nbar_sg", "Adj", "Nbar_sg"),
    ("Nbar_pl", "Adj", "Nbar_pl"),
    ("Nbar_sg", "N_sg", None),   # unary-like terminal promotion handled separately
    ("Nbar_pl", "N_pl", None),

    ("VP_sg", "Vt_sg", "NP_sg"),
    ("VP_sg", "Vt_sg", "NP_pl"),
    ("VP_pl", "Vt_pl", "NP_sg"),
    ("VP_pl", "Vt_pl", "NP_pl"),

    ("S", "NP_sg", "VP_sg"),
    ("S", "NP_pl", "VP_pl"),
]

# Unary promotions
unary_rules = [
    ("Nbar_sg", "N_sg"),
    ("Nbar_pl", "N_pl"),
    ("VP_sg", "Vt_sg"),
    ("VP_pl", "Vt_pl"),
]

class ParseTree:
    def __init__(self, label, children=None, word=None):
        self.label = label
        self.children = children if children is not None else []
        self.word = word

    def pretty(self, indent=0):
        space = "  " * indent
        if self.word is not None:
            return f"{space}({self.label} {self.word})"
        else:
            child_str = "\n".join(child.pretty(indent + 1) for child in self.children)
            return f"{space}({self.label}\n{child_str}\n{space})"

def add_unary_closure(cell):
    changed = True
    while changed:
        changed = False
        existing_labels = list(cell.keys())
        for parent, child in unary_rules:
            if child in cell and parent not in cell:
                cell[parent] = []
                for tree in cell[child]:
                    cell[parent].append(ParseTree(parent, [tree]))
                changed = True

def parse_sentence(sentence):
    tokens = tokenize(sentence)
    n = len(tokens)
    chart = [[defaultdict(list) for _ in range(n + 1)] for _ in range(n)]

    # Fill lexical entries
    for i, tok in enumerate(tokens):
        if tok in lexicon:
            for cat in lexicon[tok]:
                chart[i][i + 1][cat].append(ParseTree(cat, word=tok))
        add_unary_closure(chart[i][i + 1])

    # CKY-style combination
    for span in range(2, n + 1):
        for i in range(n - span + 1):
            j = i + span
            for k in range(i + 1, j):
                left_cell = chart[i][k]
                right_cell = chart[k][j]

                for parent, left, right in binary_rules:
                    # Handle true binary rules only
                    if right is None:
                        continue
                    if left in left_cell and right in right_cell:
                        for left_tree in left_cell[left]:
                            for right_tree in right_cell[right]:
                                chart[i][j][parent].append(ParseTree(parent, [left_tree, right_tree]))
                add_unary_closure(chart[i][j])

    return tokens, chart[0][n]

# ------------------------------------------------------------
# Part 5: NLP task (23.6) - Sentiment classification
# ------------------------------------------------------------

positive_words = {
    "good", "great", "excellent", "happy", "love", "like", "amazing", "wonderful", "clear", "useful"
}
negative_words = {
    "bad", "terrible", "awful", "sad", "hate", "boring", "confusing", "poor", "slow", "useless"
}

def sentiment_classify(text):
    tokens = tokenize(text)
    pos = sum(1 for t in tokens if t in positive_words)
    neg = sum(1 for t in tokens if t in negative_words)

    if pos > neg:
        label = "positive"
    elif neg > pos:
        label = "negative"
    else:
        label = "neutral"

    return {
        "tokens": tokens,
        "positive_hits": pos,
        "negative_hits": neg,
        "label": label
    }

# ------------------------------------------------------------
# Demonstration / output
# ------------------------------------------------------------

print("=" * 80)
print("NATURAL LANGUAGE PROCESSING SYSTEM")
print("=" * 80)

# Part 1
print("\nPART 1: LANGUAGE MODELING (23.1)")
test_sentences = [
    "the cat sees the mouse",
    "the mouse reads the teacher",
    "a happy student likes the book"
]

for s in test_sentences:
    prob = sentence_probability(s, alpha=1.0)
    logp = sentence_log_probability(s, alpha=1.0)
    print(f"\nSentence: {s}")
    print(f"Probability: {prob:.10f}")
    print(f"Log probability: {logp:.6f}")

# Part 2
print("\n" + "=" * 80)
print("PART 2: GRAMMAR AND PARSING (23.2–23.3)")
print("=" * 80)

parse_examples = [
    "the cat chases the mouse",
    "a happy student reads the book"
]

for s in parse_examples:
    tokens, result = parse_sentence(s)
    print(f"\nSentence: {s}")
    if "S" in result and len(result["S"]) > 0:
        print(f"Number of parses: {len(result['S'])}")
        print("First parse:")
        print(result["S"][0].pretty())
    else:
        print("No complete parse found.")

# Part 3
print("\n" + "=" * 80)
print("PART 3: AUGMENTED GRAMMAR (23.4)")
print("=" * 80)

agreement_tests = [
    "the cat chases the mouse",   # should parse
    "the cats chases the mouse",  # should fail because 'cats' not in lexicon and agreement mismatch
    "the teachers read the book"  # should parse (plural subject + plural/base verb)
]

for s in agreement_tests:
    tokens, result = parse_sentence(s)
    success = "S" in result and len(result["S"]) > 0
    print(f"Sentence: {s}")
    print("Parses successfully?" , success)

# Part 4
print("\n" + "=" * 80)
print("PART 4: HANDLING AMBIGUITY (23.5)")
print("=" * 80)

ambiguous_sentences = [
    "the old teacher reads the book",
    "the teacher book the students",
    "the old book"
]

for s in ambiguous_sentences:
    tokens, result = parse_sentence(s)
    print(f"\nSentence: {s}")
    if len(result) == 0:
        print("No parse found.")
        continue

    all_labels = list(result.keys())
    print("Top labels found:", all_labels)

    if "S" in result:
        print(f"Number of sentence-level parses: {len(result['S'])}")
        for idx, tree in enumerate(result["S"][:3], start=1):
            print(f"Parse {idx}:")
            print(tree.pretty())

    # Also show fragment ambiguities when full sentence parse fails
    for label in result:
        if label != "S" and len(result[label]) > 1:
            print(f"Ambiguity also found for label {label}: {len(result[label])} parses")

print("\nDiscussion:")
print("- 'book' can be a noun or a verb in the lexicon.")
print("- 'old' can be an adjective or a noun in the lexicon.")
print("- This creates structural ambiguity or lexical ambiguity in some examples.")

# Part 5
print("\n" + "=" * 80)
print("PART 5: NLP TASK (23.6) - SENTIMENT CLASSIFICATION")
print("=" * 80)

task_examples = [
    "I love this clear and useful explanation",
    "This system is confusing and slow",
    "The book is small"
]

for text in task_examples:
    result = sentiment_classify(text)
    print(f"\nText: {text}")
    print("Tokens:", result["tokens"])
    print("Positive hits:", result["positive_hits"])
    print("Negative hits:", result["negative_hits"])
    print("Predicted label:", result["label"])

print("\n" + "=" * 80)
print("SUMMARY")
print("=" * 80)
print("- Built a bigram language model to assign sentence probabilities.")
print("- Defined a grammar and parsed sentences into structured trees.")
print("- Added agreement-style constraints through augmented grammar categories.")
print("- Analyzed lexical and structural ambiguity.")
print("- Implemented an NLP task: simple sentiment classification.")
print("=" * 80)

# Build a Natural Language Processing System

## Overview
This notebook builds a small natural language processing system that handles several core NLP tasks:

- processing raw text input
- assigning probabilities to sentences with a language model
- defining grammar rules
- parsing sentences into structure
- adding grammar constraints
- analyzing ambiguity
- performing a simple NLP task: sentiment classification

The system is intentionally compact and interpretable so that each part of the NLP pipeline can be clearly explained.

---

## Part 1: Language Modeling (23.1)

### Goal
The language model assigns probabilities to sentences.

### Model design
The notebook uses a **bigram language model**. A bigram model estimates the probability of each word conditioned on the previous word:

\[
P(w_1, w_2, \dots, w_n) \approx \prod_{i=1}^{n} P(w_i \mid w_{i-1})
\]

Special start and end symbols are added:

- `<s>`
- `</s>`

This allows the model to represent sentence boundaries.

### Smoothing
Laplace smoothing is used so that unseen word pairs still receive nonzero probability.

### What the model does
For each test sentence, the notebook computes:

- sentence probability
- sentence log probability

A higher probability means the sentence is more compatible with the training corpus.

---

## Part 2: Grammar and Parsing (23.2–23.3)

### Goal
The parsing component maps a sentence into a structured representation.

### Grammar rules
The grammar includes categories such as:

- `S` for sentence
- `NP` for noun phrase
- `VP` for verb phrase
- `Det` for determiner
- `N` for noun
- `Vt` for transitive verb
- `Adj` for adjective

The grammar is implemented with explicit lexical entries and phrase-structure rules.

Examples of rules include:

\[
S \rightarrow NP\ VP
\]

\[
NP \rightarrow Det\ Nbar
\]

\[
Nbar \rightarrow Adj\ Nbar
\]

\[
VP \rightarrow Vt\ NP
\]

### Parsing method
The notebook uses a custom CKY-style chart parser. This parser builds larger constituents from smaller ones and stores possible parses for each span.

### Parsing results
For sentences that fit the grammar, the parser returns structured parse trees. These trees show how the sentence is built compositionally from words and phrase categories.

---

## Part 3: Augmented Grammar (23.4)

### Goal
The grammar is extended with constraints or features.

### Features used
The notebook adds a simple **number agreement** system:

- singular noun phrases
- plural noun phrases
- singular verb phrases
- plural verb phrases

This allows the parser to enforce subject-verb agreement.

For example:
- singular subject + singular verb can parse
- plural subject + singular verb should fail

### Why this matters
A plain context-free grammar may accept many syntactically unrealistic structures. Adding features improves grammatical precision and makes parsing more linguistically informed.

---

## Part 4: Handling Ambiguity (23.5)

### Goal
Analyze ambiguous sentences and discuss multiple interpretations.

### Types of ambiguity in the system
The notebook demonstrates **lexical ambiguity** by allowing words like:

- `book` to be both a noun and a verb
- `old` to be both an adjective and a noun

This means the parser may assign different structures depending on which category the word takes.

### Why ambiguity matters
Natural language often allows multiple interpretations. A sentence may be ambiguous because:
- a word has multiple parts of speech
- a phrase can attach in more than one way
- multiple parse trees are licensed by the grammar

The parser exposes this by returning multiple possible parses when available.

---

## Part 5: NLP Task (23.6)

### Task chosen
The notebook implements **sentiment classification**.

### How it works
The classifier uses a simple lexicon-based approach:

- count positive words
- count negative words
- compare totals

If positive hits exceed negative hits, the text is labeled **positive**.  
If negative hits exceed positive hits, the text is labeled **negative**.  
Otherwise, it is labeled **neutral**.

### Why this qualifies as an NLP task
This is a real text analysis task because it takes raw text and produces a semantic label. Although it is simple, it demonstrates end-to-end text processing and classification.

---

## Raw text processing

The system begins by tokenizing raw text:
- lowercasing
- separating words and punctuation
- converting strings into token sequences

This preprocessing step is necessary for:
- language modeling
- parsing
- text classification

---

## What each component contributes

### Language model
Helps estimate how likely a sentence is under the learned distribution.

### Grammar and parser
Reveal the internal structure of a sentence.

### Augmented grammar
Adds agreement constraints to improve grammatical validity.

### Ambiguity analysis
Shows that language can support multiple interpretations.

### Sentiment classifier
Performs a downstream NLP task on raw text.

---

## System limitations
This is a small educational NLP system, so it has several limitations:

- the training corpus is small
- the language model is only bigram-based
- the grammar covers only a small subset of English
- ambiguity handling is not probabilistically ranked
- the sentiment classifier is lexicon-based and simplistic

Even so, the notebook demonstrates the core ideas required:
- language modeling
- grammar
- parsing
- augmented grammar
- ambiguity analysis
- an NLP task

This makes it a complete small-scale NLP pipeline.

## Reflection Questions

### How did structure improve understanding?
Structure improved understanding by turning a flat sequence of words into organized linguistic units such as noun phrases, verb phrases, and full sentences. This made it possible to see who was acting, what action was happening, and what object was involved. Without structure, the system would only treat the sentence as a string of tokens. Parsing added a level of interpretation that helped the system represent relationships between words rather than just their order.

---

### Where did ambiguity cause problems?
Ambiguity caused problems when a word or phrase could reasonably be interpreted in more than one way. In this system, lexical ambiguity appeared with words like `book`, which could be either a noun or a verb, and `old`, which could be either an adjective or a noun. This made parsing more difficult because multiple structures became possible. In larger systems, such ambiguity can lead to incorrect interpretations if the parser has no strong way to prefer the intended meaning.

---

### How did probabilistic methods help?
Probabilistic methods helped by allowing the system to rank or evaluate language rather than only accept or reject it. The bigram language model assigned probabilities to sentences, which made it possible to compare which sentences looked more natural under the training corpus. In general, probabilistic methods are useful because natural language is uncertain and ambiguous, and probability provides a way to prefer more plausible interpretations instead of treating all possibilities as equal.

---

### What limitations remain in your system?
Several limitations remain in the system. The language model is very simple and only uses bigram context, so it cannot capture long-range dependencies well. The grammar is small and only covers a limited subset of sentence patterns. The ambiguity analysis is illustrative but not deeply resolved, since the system does not use a full probabilistic parser to rank competing parse trees. The sentiment classifier is also basic because it relies only on word lists and does not understand context, negation, sarcasm, or deeper meaning. Overall, the system demonstrates key ideas clearly, but it is still a simplified educational NLP pipeline rather than a full real-world solution.